# 01 — Coleta e ingestão dos dados

**TCC UNIP 2026** · Etapa 1 de 9 do pipeline · Seção 4.5.1 do trabalho escrito.

**Objetivo:** baixar, pela isic-cli, as imagens e os metadados das sete classes diagnósticas da coleção 65 (ISIC 2019).

**Entradas:**
- API pública do ISIC Archive (coleção 65)

**Saídas:**
- `/content/data/isic/<classe>/` — imagens e `metadata.csv` de cada classe

Antes de executar: no Colab, selecione *Ambiente de execução → Alterar tipo de ambiente de execução → GPU* (necessário a partir do notebook 05). As saídas exibidas foram preservadas da execução original do notebook único.

In [ ]:
# Parâmetro: True usa o Google Drive para guardar os arquivos entre notebooks;
# False usa só o disco da sessão atual. Quando o notebook é executado pelo
# 00_executar_todos.ipynb, vale o valor definido lá.
if "USAR_DRIVE" not in globals():
    USAR_DRIVE = True


In [ ]:
# ============================================================
# PERSISTÊNCIA ENTRE NOTEBOOKS (Google Drive)
# ============================================================
# Cada notebook roda em uma sessão própria do Colab. Para que os arquivos gerados por um
# notebook fiquem disponíveis para os seguintes, a pasta /content/data é ligada a uma
# pasta no Google Drive. O valor de USAR_DRIVE é definido na célula anterior.
PASTA_DRIVE = "/content/drive/MyDrive/TCC_UNIP/data"

import os
if USAR_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs(PASTA_DRIVE, exist_ok=True)
    if not os.path.islink("/content/data"):
        if os.path.isdir("/content/data"):
            if os.listdir("/content/data"):
                raise RuntimeError("/content/data já contém arquivos: mova-os para a pasta do Drive antes de continuar.")
            os.rmdir("/content/data")
        os.symlink(PASTA_DRIVE, "/content/data")
print("Pasta de dados:", os.path.realpath("/content/data"))


In [ ]:
# 1. Instalar
!pip install isic-cli --quiet

In [ ]:
!isic collection list

If you have been granted special permissions, logging in with `isic user login` might return more data.

┏━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━━┳━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ ID  ┃ Name                      ┃ Public ┃ Pinned ┃ Locked ┃ DOI             ┃
┡━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━━╇━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ 471 │ 15 Exemplar               │ True   │ False  │ True   │ 10.34970/270976 │
│     │ Infundibulocystic Basal   │        │        │        │                 │
│     │ Cell Carcinomas           │        │        │        │                 │
│ 487 │ BCC with histopathologic  │ True   │ False  │ False  │ None            │
│     │ subtype                   │        │        │        │                 │
│ 249 │ BCN20000                  │ True   │ True   │ False  │ None            │
│ 410 │ BRAAFF-Annotated Acral    │ True   │ False  │ True   │ 10.34970/669187 │
│     │ Lesions Dataset (BALD)    │        │        │        │                 │
│ 40

In [ ]:
import subprocess
import os
import torch
import torch.nn as nn

In [ ]:
CLASSES_ISIC = {
    "melanoma": 'diagnosis_2:"Malignant melanocytic proliferations (Melanoma)"',
    "nevo_melanocitico": 'diagnosis_3:"Nevus"',
    "carcinoma_basocelular": 'diagnosis_3:"Basal cell carcinoma"',
    "ceratose_actinica": 'diagnosis_3:"Solar or actinic keratosis"',
    "ceratose_seborreica": 'diagnosis_3:"Seborrheic keratosis"',
    "dermatofibroma": 'diagnosis_3:"Dermatofibroma"',
    "lesao_vascular": 'diagnosis_2:"Benign soft tissue proliferations - Vascular"',
}

In [ ]:
DESTINO_BASE = "/content/data/isic"
#LIMITE_POR_CLASSE = 70000
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
#TAMANHO_PAGINA = 100  # a API pagina os resultados

In [ ]:
def rodar_comando(comando, descricao):
    print(f"    $ {' '.join(comando)}")
    resultado = subprocess.run(comando, capture_output=True, text=True)
    if resultado.returncode != 0:
        print(f"    [ERRO em {descricao}]")
        print(resultado.stderr[-800:])
        return False
    print(f"    [OK] {descricao}")
    return True

In [ ]:
def baixar_classe(nome_pasta: str, query: str):
    destino = os.path.join(DESTINO_BASE, nome_pasta)
    os.makedirs(destino, exist_ok=True)
    caminho_csv = os.path.join(destino, "metadata.csv")

    print(f"\n>>> Classe: {nome_pasta}")

    # 1. Baixar todas as imagens da classe na coleção ISIC 2019 (ID 65)
    comando_imagens = [
        "isic", "image", "download",
        "--collections", "65",
        "--search", query,
        destino,
    ]
    ok_imagens = rodar_comando(comando_imagens, "download de imagens")

    # 2. Baixar os metadados correspondentes
    comando_metadata = [
        "isic", "metadata", "download",
        "--collections", "65",
        "--search", query,
        caminho_csv,
    ]
    ok_metadata = rodar_comando(comando_metadata, "download de metadados")

    return ok_imagens, ok_metadata

In [ ]:
if __name__ == "__main__":
    resumo = []
    for nome_pasta, query in CLASSES_ISIC.items():
        ok_img, ok_meta = baixar_classe(nome_pasta, query)
        resumo.append((nome_pasta, ok_img, ok_meta))

    print("\n" + "=" * 50)
    print("RESUMO FINAL")
    print("=" * 50)
    for nome, ok_img, ok_meta in resumo:
        status_img = "OK" if ok_img else "FALHOU"
        status_meta = "OK" if ok_meta else "FALHOU"
        print(f"{nome}: imagens={status_img} | metadata={status_meta}")


>>> Classe: melanoma
    $ isic image download --collections 65 --search diagnosis_2:"Malignant melanocytic proliferations (Melanoma)" /content/data/isic/melanoma
    [OK] download de imagens
    $ isic metadata download --collections 65 --search diagnosis_2:"Malignant melanocytic proliferations (Melanoma)" /content/data/isic/melanoma/metadata.csv
    [ERRO em download de metadados]
Usage: isic metadata download [OPTIONS]
Try 'isic metadata download --help' for help.

Error: Got unexpected extra argument (/content/data/isic/melanoma/metadata.csv)


>>> Classe: nevo_melanocitico
    $ isic image download --collections 65 --search diagnosis_3:"Nevus" /content/data/isic/nevo_melanocitico
    [OK] download de imagens
    $ isic metadata download --collections 65 --search diagnosis_3:"Nevus" /content/data/isic/nevo_melanocitico/metadata.csv
    [ERRO em download de metadados]
Usage: isic metadata download [OPTIONS]
Try 'isic metadata download --help' for help.

Error: Got unexpected extra 